# 02 - Hyperparameter search

Random search for all five models, **fold 0 only, at full n, 30
configurations each**.

**Why one fold rather than a subsample.** Both save time; they are not
equivalent. Dropping from five folds to one adds selection *noise*, which is
symmetric across models and shows up as a less precise ranking. Subsampling
rows introduces *bias*: regularisation optima shift with sample size, and they
shift unequally across architectures, so the model whose optimum moves most is
penalised. Tuning Neural Fine-Gray on less data than its competitors would
risk beating the closest published comparator because it was tuned worse -
precisely the failure the fairness rules exist to prevent.

The cost of that choice is borne by cell 7, which checks the top NFG
configurations transport to a second fold.

**Never touches the temporal or exploratory windows.**

In [1]:
# =====================================================================
# CONFIGURATION - the only cell to edit
# =====================================================================
from pathlib import Path

REPO_ROOT    = Path(r"C:\Users\olf3ph\Documents")          # repository root; leave "" to auto-detect
FEATURES_DIR = Path(r"C:\Users\olf3ph\Documents\FEATURES")          # feature-builder output
RESULTS_DIR  = Path(r"C:\Users\olf3ph\Documents\RESULTS")          # benchmark parquet + logs
NFG_PATH     = Path(r"C:\Users\olf3ph\Documents\NeuralFineGray")

T_EVAL = [1.0, 3.0, 5.0]          # years
FOLDS  = [0, 1, 2, 3, 4]

# =====================================================================
# Nothing below this line needs editing.
# =====================================================================
import sys, os, json, time
import numpy as np
import pandas as pd

_here = Path.cwd()
if not str(REPO_ROOT):
    # Notebook may be run from notebooks/ or from the repository root.
    REPO_ROOT = _here.parent if (_here / "io_helpers.py").exists() else _here
REPO_ROOT = Path(REPO_ROOT).resolve()

sys.path.insert(0, str(REPO_ROOT / "notebooks"))

#os.environ["NFG_PATH"] = r"C:\Users\olf3ph\Documents\NeuralFineGray"

if str(NFG_PATH):
    os.environ["NFG_PATH"] = str(NFG_PATH)
    assert NFG_PATH.exists(), f"NFG_PATH does not exist: {NFG_PATH}"

import io_helpers as ioh
ioh.add_rp_core_to_path(REPO_ROOT)

FEATURES_DIR = Path(FEATURES_DIR)
RESULTS_DIR = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_DIR.exists(), f"FEATURES_DIR does not exist: {FEATURES_DIR}"
ioh.log("configuration", repo=REPO_ROOT, features=FEATURES_DIR,
        results=RESULTS_DIR)

# What the feature builder actually produced. The held_out column is a
# reminder, not a permission: the loader refuses those sets outright.
_avail = ioh.available_sets(FEATURES_DIR)
print(_avail.to_string(index=False))

[2026-08-21 13:03:01] configuration  repo=C:\Users\olf3ph\Documents  features=C:\Users\olf3ph\Documents\FEATURES  results=C:\Users\olf3ph\Documents\RESULTS
     applied_to  imputation  held_out  onehot  embedding
 dev_fold0_test           1     False    True       True
 dev_fold0_test           2     False    True       True
 dev_fold0_test           3     False    True       True
 dev_fold0_test           4     False    True       True
 dev_fold0_test           5     False    True       True
dev_fold0_train           1     False    True       True
dev_fold0_train           2     False    True       True
dev_fold0_train           3     False    True       True
dev_fold0_train           4     False    True       True
dev_fold0_train           5     False    True       True
 dev_fold1_test           1     False    True       True
 dev_fold1_test           2     False    True       True
 dev_fold1_test           3     False    True       True
 dev_fold1_test           4     False    True 

## 1. Search spaces, side by side

All five declared in one place, so the budgets are visibly comparable rather
than comparable by assertion. Every grid is far larger than the 30-trial
budget, so no model is quietly searched exhaustively while another is sampled.

**Selection criterion** is held-out log-likelihood where the model provides
one, and integrated Brier score otherwise. DeepHit is the exception: its
training loss mixes likelihood with a ranking term and is not a likelihood, so
it is selected on IBS. Which criterion each model used is recorded in the trial
log and printed at selection time - comparing a likelihood-selected model with
a Brier-selected one is fine, but only if the reader knows.

In [2]:
from benchmark import SEARCH_SPACES, make_cv_folds, random_search, \
    search_space_table
from benchmark.deep import (DeepConfig, DSMBaseline, DeepHitBaseline,
                            NeuralFineGrayBaseline, available_deep_baselines)

BUDGET = 30
MAX_EPOCHS = 200          # variants A/B; early stopping usually ends sooner
TUNING_FOLD = 0
TUNING_IMPUTATION = 1
MODELS = ["variant_a", "variant_b", "deephit", "neural_fine_gray", "dsm"]

table = search_space_table()
print(table[["model", "criterion", "n_hyperparameters", "grid_size"]]
      .to_string(index=False))
print()
for name in MODELS:
    sp = SEARCH_SPACES[name]
    print(f"{name:18s} budget {BUDGET:3d} of {sp.size():6d} configurations "
          f"({100 * BUDGET / sp.size():.1f}% of the grid)  "
          f"criterion={sp.criterion}")
    print(f"{'':18s} {', '.join(sorted(sp.grid))}")

# Deep baselines must actually be importable before hours are spent.
for name, (ok, why) in available_deep_baselines().items():
    print(f"\n{name:18s} {'available' if ok else 'UNAVAILABLE: ' + why}")

           model criterion  n_hyperparameters  grid_size
         deephit       ibs                  8       8748
             dsm    loglik                  6       1080
neural_fine_gray    loglik                  6        648
       variant_a    loglik                  5        324
       variant_b    loglik                  6        972

variant_a          budget  30 of    324 configurations (9.3% of the grid)  criterion=loglik
                   dropout, encoder_lr, hidden_dims, latent_dim, spline_lr
variant_b          budget  30 of    972 configurations (3.1% of the grid)  criterion=loglik
                   dropout, encoder_lr, head_lr, hidden_dims, latent_dim, spline_lr
deephit            budget  30 of   8748 configurations (0.3% of the grid)  criterion=ibs
                   alpha, batch_size, dropout, head, hidden, lr, num_durations, sigma
neural_fine_gray   budget  30 of    648 configurations (4.6% of the grid)  criterion=loglik
                   act, batch_size, dropout, he

## 2. Load fold 0, and build the CV folds the harness wants

Two representations: the neural and deep models consume `embedding`
(`[continuous | integer codes]`), the classical ones `onehot`. Both come from
the same fitted `FeatureSpec` - identical information, never re-encoded.

In [3]:
tune_train, tune_val = ioh.load_fold(FEATURES_DIR, TUNING_FOLD,
                                     TUNING_IMPUTATION, "embedding")
print(ioh.describe(tune_train))
print(ioh.describe(tune_val))

N_CONTINUOUS = tune_train["n_continuous"]
CARDINALITIES = tune_train["cardinalities"]
COLUMNS = tune_train["columns"]
print(f"\nembedding: {N_CONTINUOUS} continuous + {len(CARDINALITIES)} "
      f"categorical (cardinalities {CARDINALITIES})")

# The harness takes a list of {"train":, "validate":} dicts. One entry here,
# because this is a single-fold search by design.
folds = [{"fold": TUNING_FOLD,
          "train": ioh.as_cell_data(tune_train),
          "validate": ioh.as_cell_data(tune_val)}]

# Explicit guard: the search must never see a held-out window.
ioh.assert_not_held_out(tune_train["applied_to"], tune_val["applied_to"])

dev_fold0_train m1 [embedding]  37,821 rows x 43 cols   cause1 3,525  cause2 4,431  censored 29,865
dev_fold0_test m1 [embedding]  9,469 rows x 43 cols   cause1 913  cause2 1,073  censored 7,483

embedding: 16 continuous + 27 categorical (cardinalities [2, 2, 2, 2, 2, 2, 5, 2, 13, 5, 11, 6, 6, 3, 5, 3, 3, 3, 3, 3, 2, 7, 7, 7, 3, 3, 3])


## 3. Timing probe

One fit per model at full n, before committing to 30 configurations each.
Neural Fine-Gray is the expensive one by a wide margin - roughly 1,210 s per
fit at 37,000 rows against 25 s for DeepHit and 10 s for DSM - so its 30
configurations dominate the total.

Its cost is not linear in rows and not a fixed schedule: the number of
iterations before early stopping fires grows with n. That is why the probe
measures rather than extrapolates.

In [4]:
def make_model(name, config, seed):
    """One model from a sampled config - the factory the harness calls."""
    if name == "deephit":
        return DeepHitBaseline(columns=COLUMNS, n_continuous=N_CONTINUOUS,
                               cardinalities=CARDINALITIES, seed=seed,
                               config=DeepConfig(**config))
    if name == "neural_fine_gray":
        return NeuralFineGrayBaseline(columns=COLUMNS,
                                      n_continuous=N_CONTINUOUS,
                                      cardinalities=CARDINALITIES, seed=seed,
                                      config=DeepConfig(**config))
    if name == "dsm":
        return DSMBaseline(columns=COLUMNS, n_continuous=N_CONTINUOUS,
                           cardinalities=CARDINALITIES, seed=seed,
                           config=DeepConfig(**config))
    if name in ("variant_a", "variant_b"):
        # The proposed models, through the same protocol and the same search
        # procedure as their comparators. VariantBaseline wraps rp_core's own
        # VariantA/VariantB, RCSBasis, train_model and cif.predict_cif - no
        # estimation logic is reimplemented here.
        return ioh.VariantBaseline(variant=name, n_continuous=N_CONTINUOUS,
                                   cardinalities=CARDINALITIES,
                                   columns=COLUMNS, seed=seed,
                                   max_epochs=MAX_EPOCHS, **config)
    raise KeyError(f"no factory for {name!r}")


probe_seconds = {}
for name in MODELS:
    cfg = SEARCH_SPACES[name].sample(np.random.default_rng(0))
    try:
        with ioh.timed(f"probe {name}"):
            t0 = time.time()
            m = make_model(name, cfg, seed=0)
            m.fit(tune_train["X"], tune_train["time"], tune_train["event"])
            probe_seconds[name] = time.time() - t0
    except Exception as exc:
        print(f"{name:18s} PROBE FAILED: {type(exc).__name__}: {exc}")
        continue
    total = probe_seconds[name] * BUDGET
    print(f"{name:18s} {ioh.human(probe_seconds[name]):>8s} per fit  ->  "
          f"{ioh.human(total):>8s} for {BUDGET} configurations")

if probe_seconds:
    print(f"\nPROJECTED TOTAL: "
          f"{ioh.human(sum(probe_seconds.values()) * BUDGET)}")

[2026-08-21 13:03:12] START  probe variant_a
[2026-08-21 13:03:30] DONE   probe variant_a  elapsed=18s
variant_a               18s per fit  ->      9.1m for 30 configurations
[2026-08-21 13:03:30] START  probe variant_b
[2026-08-21 13:03:45] DONE   probe variant_b  elapsed=14s
variant_b               14s per fit  ->      7.2m for 30 configurations
[2026-08-21 13:03:45] START  probe deephit
[2026-08-21 13:04:00] DONE   probe deephit  elapsed=15s
deephit                 15s per fit  ->      7.7m for 30 configurations
[2026-08-21 13:04:00] START  probe neural_fine_gray
[2026-08-21 13:07:08] DONE   probe neural_fine_gray  elapsed=3.1m
neural_fine_gray       3.1m per fit  ->     1.56h for 30 configurations
[2026-08-21 13:07:08] START  probe dsm
[2026-08-21 13:07:25] DONE   probe dsm  elapsed=17s
dsm                     17s per fit  ->      8.7m for 30 configurations

PROJECTED TOTAL: 2.11h


## 4. Run the search, one model at a time

Trials are written to parquet **after each model finishes**, so an interruption
during Neural Fine-Gray - the long one - does not lose the cheaper models that
already completed. A failed trial is recorded rather than dropped: a silently
skipped configuration would make the reported budget a lie.

In [5]:
TRIALS_PATH = RESULTS_DIR / "tuning_trials.parquet"
results = {}

for name in MODELS:
    ioh.log(f"searching {name}", budget=BUDGET,
            criterion=SEARCH_SPACES[name].criterion)
    try:
        with ioh.timed(f"random search: {name}"):
            results[name] = random_search(
                name,
                factory=lambda cfg, seed, _n=name: make_model(_n, cfg, seed),
                folds=folds, t_eval=np.asarray(T_EVAL), budget=BUDGET,
                seed=42, output_path=TRIALS_PATH,
                window=np.array(["development"]),
                imputation=TUNING_IMPUTATION, verbose=True)
        r = results[name]
        ioh.log(f"{name} done", best=f"{r.best_score:+.5f}",
                evaluated=r.n_evaluated)
    except Exception as exc:
        ioh.log(f"{name} FAILED: {type(exc).__name__}: {exc}")

ioh.log("trials written", path=TRIALS_PATH)

[2026-08-21 13:07:25] searching variant_a  budget=30  criterion=loglik
[2026-08-21 13:07:25] START  random search: variant_a
  [variant_a] trial 1/30     -0.89817    10.6s  {'hidden_dims': [128, 64], 'latent_dim': 128, 'dropout': 0.2, 'encoder_lr': 0.001, 'spline_lr': 0.01}
  [variant_a] trial 2/30     -0.90741    25.1s  {'hidden_dims': [512, 256], 'latent_dim': 32, 'dropout': 0.2, 'encoder_lr': 0.0003, 'spline_lr': 0.003}
  [variant_a] trial 3/30     -0.89625     8.4s  {'hidden_dims': [256, 128], 'latent_dim': 128, 'dropout': 0.2, 'encoder_lr': 0.003, 'spline_lr': 0.03}
  [variant_a] trial 4/30     -0.90073    12.8s  {'hidden_dims': [512, 256], 'latent_dim': 64, 'dropout': 0.0, 'encoder_lr': 0.003, 'spline_lr': 0.01}
  [variant_a] trial 5/30     -0.89770     9.0s  {'hidden_dims': [256, 128], 'latent_dim': 64, 'dropout': 0.0, 'encoder_lr': 0.003, 'spline_lr': 0.03}
  [variant_a] trial 6/30     -0.89783    11.6s  {'hidden_dims': [256, 128], 'latent_dim': 64, 'dropout': 0.3, 'encoder_lr'

C:\Users\olf3ph\Documents\rp_core\.venv\Lib\site-packages\pycox\preprocessing\discretization.py:37: UserWarning: cuts are not unique, continue with 98 cuts instead of 100
  warnings.warn(f"cuts are not unique, continue with {len(cuts)} cuts instead of {num}")


  [deephit] trial 1/30     -0.12210    18.1s  {'hidden': [128, 64], 'head': [128], 'dropout': 0.2, 'lr': 0.001, 'batch_size': 256, 'num_durations': 100, 'alpha': 0.0, 'sigma': 0.5}


C:\Users\olf3ph\Documents\rp_core\.venv\Lib\site-packages\pycox\preprocessing\discretization.py:37: UserWarning: cuts are not unique, continue with 98 cuts instead of 100
  warnings.warn(f"cuts are not unique, continue with {len(cuts)} cuts instead of {num}")


  [deephit] trial 2/30     -0.05373    11.6s  {'hidden': [128, 64], 'head': [32], 'dropout': 0.2, 'lr': 0.003, 'batch_size': 512, 'num_durations': 100, 'alpha': 0.5, 'sigma': 0.5}
  [deephit] trial 3/30     -0.10392    14.1s  {'hidden': [256, 128], 'head': [32], 'dropout': 0.3, 'lr': 0.001, 'batch_size': 256, 'num_durations': 50, 'alpha': 0.0, 'sigma': 0.5}
  [deephit] trial 4/30     -0.05360    10.2s  {'hidden': [512, 256], 'head': [64], 'dropout': 0.1, 'lr': 0.003, 'batch_size': 256, 'num_durations': 50, 'alpha': 0.2, 'sigma': 0.1}


C:\Users\olf3ph\Documents\rp_core\.venv\Lib\site-packages\pycox\preprocessing\discretization.py:37: UserWarning: cuts are not unique, continue with 98 cuts instead of 100
  warnings.warn(f"cuts are not unique, continue with {len(cuts)} cuts instead of {num}")


... (9 outputs left out)

C:\Users\olf3ph\Documents\rp_core\.venv\Lib\site-packages\pycox\preprocessing\discretization.py:37: UserWarning: cuts are not unique, continue with 98 cuts instead of 100
  warnings.warn(f"cuts are not unique, continue with {len(cuts)} cuts instead of {num}")


  [deephit] trial 21/30     -0.06603    22.4s  {'hidden': [512, 256], 'head': [64], 'dropout': 0.1, 'lr': 0.001, 'batch_size': 256, 'num_durations': 100, 'alpha': 0.0, 'sigma': 0.1}


C:\Users\olf3ph\Documents\rp_core\.venv\Lib\site-packages\pycox\preprocessing\discretization.py:37: UserWarning: cuts are not unique, continue with 98 cuts instead of 100
  warnings.warn(f"cuts are not unique, continue with {len(cuts)} cuts instead of {num}")


  [deephit] trial 22/30     -0.05393    18.9s  {'hidden': [256, 128], 'head': [32], 'dropout': 0.0, 'lr': 0.0003, 'batch_size': 512, 'num_durations': 100, 'alpha': 0.5, 'sigma': 0.25}
  [deephit] trial 23/30     -0.05360    13.4s  {'hidden': [512, 256], 'head': [32], 'dropout': 0.3, 'lr': 0.001, 'batch_size': 512, 'num_durations': 20, 'alpha': 0.2, 'sigma': 0.5}
  [deephit] trial 24/30     -0.05346    19.9s  {'hidden': [256, 128], 'head': [64], 'dropout': 0.0, 'lr': 0.001, 'batch_size': 128, 'num_durations': 20, 'alpha': 0.5, 'sigma': 0.25}
  [deephit] trial 25/30     -0.05387    26.9s  {'hidden': [256, 128], 'head': [64], 'dropout': 0.3, 'lr': 0.0003, 'batch_size': 256, 'num_durations': 20, 'alpha': 0.2, 'sigma': 0.5}
  [deephit] trial 26/30     -0.05371    12.5s  {'hidden': [256, 128], 'head': [128], 'dropout': 0.1, 'lr': 0.003, 'batch_size': 256, 'num_durations': 20, 'alpha': 0.5, 'sigma': 0.5}
  [deephit] trial 27/30     -0.05356     8.4s  {'hidden': [128, 64], 'head': [128], 'drop

## 5. Selection

The top five per model, not just the winner. If the best two configurations
are separated by less than the spread across nearby configurations, the
selection is within noise and the winner is arbitrary - worth seeing before
committing it to the full matrix.

In [6]:
trials = pd.read_parquet(TRIALS_PATH)

for name, r in results.items():
    print(f"\n{'=' * 70}\n{name}   criterion={r.criterion}   "
          f"evaluated {r.n_evaluated}/{r.budget}")
    print(f"BEST  score {r.best_score:+.6f}")
    print(f"      {r.best_config}")
    top = (trials[(trials.model == name) & (~trials.failed)]
           .nlargest(5, "score")[["score", "score_std", "fit_seconds",
                                  "config"]])
    print("\ntop 5:")
    print(top.to_string(index=False))
    if len(top) > 1:
        gap = top.score.iloc[0] - top.score.iloc[1]
        spread = top.score.iloc[0] - top.score.iloc[-1]
        flag = "  <-- KNIFE EDGE" if spread > 0 and gap < 0.2 * spread else ""
        print(f"\ngap to runner-up {gap:.6f}, spread across top 5 "
              f"{spread:.6f}{flag}")

n_failed = int(trials.failed.sum())
if n_failed:
    print(f"\n{n_failed} failed trial(s):")
    print(trials[trials.failed][["model", "trial", "error"]].to_string(index=False))


variant_a   criterion=loglik   evaluated 30/30
BEST  score -0.895184
      {'hidden_dims': [512, 256], 'latent_dim': 128, 'dropout': 0.2, 'encoder_lr': 0.001, 'spline_lr': 0.03}

top 5:
    score  score_std  fit_seconds                                                                                                 config
-0.895184        0.0    13.428053 {'hidden_dims': [512, 256], 'latent_dim': 128, 'dropout': 0.2, 'encoder_lr': 0.001, 'spline_lr': 0.03}
-0.895754        0.0     6.689594   {'hidden_dims': [128, 64], 'latent_dim': 64, 'dropout': 0.0, 'encoder_lr': 0.003, 'spline_lr': 0.03}
-0.896247        0.0     8.433498 {'hidden_dims': [256, 128], 'latent_dim': 128, 'dropout': 0.2, 'encoder_lr': 0.003, 'spline_lr': 0.03}
-0.896296        0.0    10.783196 {'hidden_dims': [128, 64], 'latent_dim': 128, 'dropout': 0.3, 'encoder_lr': 0.0003, 'spline_lr': 0.01}
-0.896614        0.0    13.009628  {'hidden_dims': [512, 256], 'latent_dim': 64, 'dropout': 0.3, 'encoder_lr': 0.003, 'spline_lr

## 6. Write the selected configurations

`selected_configs.json` is the artifact the full development run consumes. It
records the criterion and the budget alongside each configuration, so the run
that uses it can state how the configuration was chosen without going back to
the trial log.

In [7]:
SELECTED_PATH = RESULTS_DIR / "selected_configs.json"

payload = {
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "tuning_fold": TUNING_FOLD,
    "tuning_imputation": TUNING_IMPUTATION,
    "budget": BUDGET,
    "t_eval": list(T_EVAL),
    "note": ("Single-fold search at full n. One fold adds selection noise "
             "(symmetric across models); subsampling rows would add bias "
             "(asymmetric), so the fold was cut rather than the rows."),
    "models": {name: {"config": r.best_config, "criterion": r.criterion,
                      "score": float(r.best_score),
                      "n_evaluated": int(r.n_evaluated)}
               for name, r in results.items()},
}
SELECTED_PATH.write_text(json.dumps(payload, indent=2, default=str))
ioh.log("selected configs written", path=SELECTED_PATH,
        models=len(payload["models"]))
print(json.dumps(payload, indent=2, default=str))

[2026-08-21 14:40:40] selected configs written  path=C:\Users\olf3ph\Documents\RESULTS\selected_configs.json  models=5
{
  "created": "2026-08-21 14:40:40",
  "tuning_fold": 0,
  "tuning_imputation": 1,
  "budget": 30,
  "t_eval": [
    1.0,
    3.0,
    5.0
  ],
  "note": "Single-fold search at full n. One fold adds selection noise (symmetric across models); subsampling rows would add bias (asymmetric), so the fold was cut rather than the rows.",
  "models": {
    "variant_a": {
      "config": {
        "hidden_dims": [
          512,
          256
        ],
        "latent_dim": 128,
        "dropout": 0.2,
        "encoder_lr": 0.001,
        "spline_lr": 0.03
      },
      "criterion": "loglik",
      "score": -0.8951835632324219,
      "n_evaluated": 30
    },
    "variant_b": {
      "config": {
        "hidden_dims": [
          256,
          128
        ],
        "latent_dim": 128,
        "dropout": 0.2,
        "encoder_lr": 0.001,
        "spline_lr": 0.03,
        "hea

## 7. Neural Fine-Gray transport check

Single-fold selection is noisier than five-fold. This re-fits the top three NFG
configurations on **fold 1** and checks the ranking holds.

**If the ranking reorders, the selection is within noise.** That is not a
failure of the run - it is the information the check exists to produce, and it
matters because NFG is the closest published comparator to this project's
model. Committing a configuration chosen by noise would mean any margin over
it is partly an artifact of the search. If it reorders, either average the top
few configurations' scores across both folds and take that winner, or spend
the extra folds on NFG alone and say so.

In [8]:
TRANSPORT_FOLD = 1

if "neural_fine_gray" not in results:
    print("Neural Fine-Gray was not searched; nothing to transport-check.")
else:
    import ast
    from benchmark.tuning import score_fold

    nfg_trials = (trials[(trials.model == "neural_fine_gray")
                         & (~trials.failed)].nlargest(3, "score"))
    tr1, va1 = ioh.load_fold(FEATURES_DIR, TRANSPORT_FOLD, TUNING_IMPUTATION,
                             "embedding")
    ioh.assert_not_held_out(tr1["applied_to"], va1["applied_to"])
    print(ioh.describe(tr1))

    rows = []
    for rank, (_, row) in enumerate(nfg_trials.iterrows(), start=1):
        cfg = ast.literal_eval(row["config"])
        with ioh.timed(f"NFG rank {rank} on fold {TRANSPORT_FOLD}"):
            m = make_model("neural_fine_gray", cfg, seed=1000 + rank)
            m.fit(tr1["X"], tr1["time"], tr1["event"])
            s = score_fold(m, va1["X"], va1["time"], va1["event"],
                           np.asarray(T_EVAL),
                           SEARCH_SPACES["neural_fine_gray"].criterion)
        rows.append({"rank_fold0": rank, "score_fold0": row["score"],
                     "score_fold1": s, "config": row["config"]})

    check = pd.DataFrame(rows)
    check["rank_fold1"] = check.score_fold1.rank(ascending=False).astype(int)
    print("\n" + check[["rank_fold0", "rank_fold1", "score_fold0",
                        "score_fold1"]].to_string(index=False))

    held = bool((check.rank_fold0.values == check.rank_fold1.values).all())
    winner_held = bool(check.rank_fold1.iloc[0] == 1)
    print(f"\nfull ranking held : {held}")
    print(f"winner held       : {winner_held}")
    if not winner_held:
        print("\n!! The fold-0 winner is not the fold-1 winner. The selection "
              "is within noise. Do NOT quietly keep the fold-0 choice - "
              "either average across both folds and take that winner, or run "
              "the remaining folds for NFG and report the wider search.")

dev_fold1_train m1 [embedding]  37,823 rows x 43 cols   cause1 3,553  cause2 4,403  censored 29,867
[2026-08-21 14:40:40] START  NFG rank 1 on fold 1
[2026-08-21 14:41:32] DONE   NFG rank 1 on fold 1  elapsed=51s
[2026-08-21 14:41:32] START  NFG rank 2 on fold 1
[2026-08-21 14:46:31] DONE   NFG rank 2 on fold 1  elapsed=5.0m
[2026-08-21 14:46:31] START  NFG rank 3 on fold 1
[2026-08-21 14:48:35] DONE   NFG rank 3 on fold 1  elapsed=2.1m

 rank_fold0  rank_fold1  score_fold0  score_fold1
          1           1    -0.910486    -0.911650
          2           3    -0.911163    -0.912174
          3           2    -0.911282    -0.911998

full ranking held : False
winner held       : True


## 8. Summary

**Produced:**

- `tuning_trials.parquet` - every trial: model, config, per-fold score, wall
  time, seed, and the failure reason where one applies. This is a supplementary
  table for the paper, not a debugging aid: it is what shows a reviewer the
  comparators were given a real search.
- `selected_configs.json` - the chosen configuration per model, with its
  criterion, score, and budget.

**Covers:** fold 0, imputation m = 1, 30 configurations per model, at full n.
Temporal and exploratory untouched.

**Consumed by:** the full development run, which fixes these configurations and
fits them across all folds and imputations.

**To report in the methods:** the budget and grid size per model (cell 1), the
criterion per model, and the transport-check result from cell 7.

In [9]:
print(f"tuning_trials.parquet : {TRIALS_PATH}")
print(f"selected_configs.json : {SELECTED_PATH}")
print(f"models searched       : {sorted(results)}")
print(f"trials logged         : {len(trials):,} "
      f"({int(trials.failed.sum())} failed)")
print(f"total search time     : "
      f"{ioh.human(trials.fit_seconds.sum())}")
for name, r in results.items():
    print(f"  {name:18s} {r.criterion:7s} {r.best_score:+.6f}")

tuning_trials.parquet : C:\Users\olf3ph\Documents\RESULTS\tuning_trials.parquet
selected_configs.json : C:\Users\olf3ph\Documents\RESULTS\selected_configs.json
models searched       : ['deephit', 'dsm', 'neural_fine_gray', 'variant_a', 'variant_b']
trials logged         : 150 (0 failed)
total search time     : 1.55h
  variant_a          loglik  -0.895184
  variant_b          loglik  -0.892377
  deephit            ibs     -0.053381
  neural_fine_gray   loglik  -0.910486
  dsm                loglik  -0.881652
